# EasyRAG Repo Loading Basics

## Chapter position

This chapter sits before indexing. Raw files or in-memory text become canonical `Document` objects here, so mistakes at this layer tend to echo through every later stage.

## Learning question

How do repository files become canonical `Document` objects before indexing ever starts?

## Success criteria

- create a tiny repository-shaped corpus
- load repository documents through the canonical EasyRAG entry point
- inspect the metadata that later powers grounded citations

## Source code anchors

- `easyrag.rag.indexing.loaders.load_repo_documents`
- `easyrag.rag.indexing.chunking.chunk_documents`
- `easyrag.rag.orchestrator.EasyRAG`
- `easyrag.support.async_utils.run_sync`


## Method principles

- `easyrag.rag.indexing.loaders.load_repo_documents`: This is the canonical repo discovery entry point. It filters indexable files, normalizes their text, and returns canonical `Document` objects with stable metadata.
- `easyrag.rag.indexing.chunking.chunk_documents`: This wrapper applies the chunking strategy registry across a document list. Its job is to turn document-level inputs into retrieval units while preserving the metadata needed later.
- `easyrag.rag.orchestrator.EasyRAG`: This is the public orchestration facade. It keeps loading, indexing, retrieval, and answering behind one lifecycle so the notebooks can focus on stage boundaries instead of wiring.
- `easyrag.support.async_utils.run_sync`: This is the notebook bridge into the async API. It lets synchronous notebook cells call methods like `ainsert()` or `aquery()` without making the tutorial rewrite everything around event loops.

Design reason: these anchors are chosen at the raw-input to canonical-document boundary. The notebook keeps that contract explicit because later chunking, embedding, and retrieval quality are only as trustworthy as the documents and metadata produced here.


## How the code fits together

The flow in this notebook is repo files -> canonical documents -> chunk preview -> cleanup. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the notebook establishes raw material first, then turns it into canonical documents, then inspects what survived that normalization boundary. That order makes it clear which later behaviors come from source quality and which come from later indexing or retrieval policy.


## Step 1: Import the indexing and query helpers

We will use the public `EasyRAG` API for loading and querying, and a small set of indexing helpers for chunk inspection and workspace rebuilds. `run_sync` is still useful because the orchestrator lifecycle is async even inside a notebook.


## What this cell is proving

We start by making the repo importable from the notebook runtime and loading the helpers this notebook depends on. This cell should stay quiet. It is only here to make the later examples reliable.

In [ ]:
import importlib
import json
import sys
import tempfile
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

EasyRAG = importlib.import_module("easyrag").EasyRAG
indexing = importlib.import_module("easyrag.rag.indexing")
ChunkingConfig = indexing.ChunkingConfig
chunk_documents = indexing.chunk_documents

This cell should only import symbols. The important thing to notice is the separation of roles: `EasyRAG` is the high-level entry point, while `chunk_documents()` and `rebuild_document_index()` expose the indexing mechanics more directly.


## Step 2: Create a tiny repository-shaped corpus

Instead of indexing the full EasyRAG repository, we create a small temporary repo with a `README.md` and a few files under `docs/`. That keeps the example deterministic and makes it obvious why each document was loaded.

Design reason: the notebook uses hand-shaped inputs here so the later behavior can be attributed to the stage under study instead of to noisy source material. When the examples have obvious roles and boundaries, it becomes much easier to tell whether the pipeline preserved the intended structure.


In [ ]:
repo_temp_dir = tempfile.TemporaryDirectory()
repo_root = Path(repo_temp_dir.name) / "demo_repo"
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True)

(repo_root / "README.md").write_text(
    "# Demo Repo\nThis repository explains EasyRAG indexing.\n",
    encoding="utf-8",
)
(docs_dir / "architecture.md").write_text(
    "# Architecture\nEasyRAG organizes indexing as loading, chunking, and retrieval.\n## Chunking\nStructured chunks keep section boundaries.\n",
    encoding="utf-8",
)
(docs_dir / "notes.txt").write_text(
    "Semantic chunking works well for plain text notes about embeddings and retrieval.\n",
    encoding="utf-8",
)
(docs_dir / "operations.md").write_text(
    "# Operations\nThe build script can rebuild, update, or delete index entries.\n",
    encoding="utf-8",
)

print(
    json.dumps(
        sorted(
            str(path.relative_to(repo_root))
            for path in repo_root.rglob("*")
            if path.is_file()
        ),
        indent=2,
    )
)

You should see a small file list rooted at `demo_repo/`. This is the corpus we will index. Because the directory layout looks like a repository, the same loader logic used on real docs can operate on it without any special casing.


## Step 3: Load repository documents

`EasyRAG.load_repo_documents()` is the teaching-friendly entry point for repository discovery. It applies the repo loader rules and returns canonical `Document` objects with metadata such as title, path, and `doc_id`.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
documents = EasyRAG.load_repo_documents(repo_root)

document_preview = [
    {
        "title": str(document.metadata.get("title")),
        "relative_path": str(document.metadata.get("relative_path")),
        "doc_id": str(document.metadata.get("doc_id")),
        "source_type": str(document.metadata.get("source_type")),
    }
    for document in documents
]

print(json.dumps(document_preview, indent=2))

You should see one record per loaded source file. The key observation is that indexing does not begin with anonymous text blobs. It begins with documents that already carry stable metadata, and that metadata will later show up in citations and storage records.


## Step 4: Inspect chunk boundaries before building the workspace

Chunking is where raw documents become retrieval units. We inspect chunks before writing any index so you can see which strategy EasyRAG selected and what metadata each chunk carries.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
chunks = chunk_documents(documents, config=ChunkingConfig())

chunk_preview = [
    {
        "doc_id": str(chunk.metadata.get("doc_id")),
        "title": str(chunk.metadata.get("title")),
        "chunk_id": int(chunk.metadata.get("chunk_id", 0)),
        "chunk_strategy": str(chunk.metadata.get("chunk_strategy")),
        "overlap_policy": str(chunk.metadata.get("overlap_policy")),
        "snippet": chunk.page_content[:90],
    }
    for chunk in chunks
]

print(json.dumps(chunk_preview, indent=2))

This output should show a mix of strategies. Markdown files usually become `structured` chunks, while plain text files often fall back to `sliding_window` when no semantic embedding function is available. That is a useful reminder that chunking is adaptive, not one-size-fits-all.


## Step 5: Clean up the temporary repository

We are done with the loading-side walkthrough. Cleaning up the temporary repo keeps the notebook self-contained and makes it obvious that we have not built a persistent EasyRAG workspace yet.

In [ ]:
repo_temp_dir.cleanup()

['After cleanup, the temporary repository is gone. That is expected. This notebook is about getting source material into canonical `Document` objects and inspecting the first chunk boundary decisions.\n', '\n', '

## Next steps

- Compare this repository-shaped path with [02_02_manual_document_preparation.ipynb](02_02_manual_document_preparation.ipynb) if your inputs start in memory instead of on disk.
- Continue with [02_04_normalization_and_cleaning.ipynb](02_04_normalization_and_cleaning.ipynb) to inspect cleaning and normalization directly.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the loading-stage explanation behind these paths.


## Common mistakes / debugging cues

- If a later stage looks wrong, inspect `doc_id`, logical path, title, and normalized text here first.
- Do not confuse canonical `Document` preparation with actual index writes. They are different boundaries.
- Noisy or empty documents usually create problems long before retrieval. Loading and quality checks are where you catch them cheaply.

## Takeaway

The point of this notebook is to keep `repo files -> canonical documents -> chunk preview -> cleanup` visible enough that the stage boundary stays readable. Once that boundary makes sense, the later notebooks stop feeling like disconnected tricks.

Continue with [02_02_manual_document_preparation.ipynb](02_02_manual_document_preparation.ipynb) if you want to stay in the same chapter order.